<div align="center">

<a href="https://colab.research.google.com/github/utkukose/cyber-physical-systems-NB-lecture/blob/main/exams/midterm/MID2_solar_street_light.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm option 2: A solar street light

**Cyber-Physical Systems (BIL-435 Siber Fiziksel Sistemler)**<br>Midterm project: A model and a controller for a cyber-physical system. Starter Colab notebook

[Project page](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/exams/midterm) | [Course repository](https://github.com/utkukose/cyber-physical-systems-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: A solar street light with a battery of 1.2 kWh over ten winter days. Replace the values marked `# TODO` with the data of your own system. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. The energy balance and the state of charge

*Lecture: [Week 1, Energy and power](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#energy-and-power) and [Week 2, The battery: State of charge as an integrator](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#the-battery)*

The street light has a solar panel, a battery and a light-emitting diode (LED) lamp. The panel delivers the power
$P_{pv}$ by day, and the lamp draws $P_{lamp}$ by night, both in W. The state of charge (SoC) $s$ of the battery
integrates the difference of the two:

$$\frac{ds}{dt} = \frac{\eta_c\,P_{in} - P_{out}/\eta_d}{E},$$

where $E$ is the usable energy of the battery in Wh, and $\eta_c$ and $\eta_d$ are the efficiencies of charging and
discharging. A full battery takes no more charge, and an empty battery cannot feed the lamp. The model of the panel is a
half sine wave between sunrise and sunset, scaled by the clearness of each day. The cell simulates ten winter days with
the lamp at full power from dusk to dawn and without any protection. It prints the energy balance of each day. It
also counts the dark hours and the hours of deep discharge below 15 percent, in which the battery ages fast.

In [ ]:
# TODO: the components of your system and ten days of weather
E_WH = 1200                  # usable energy of the battery (Wh): 12.8 V and 100 Ah
ETA_C, ETA_D = 0.95, 0.95    # efficiencies of charging and discharging
P_PV_PEAK = 150              # rated power of the solar panel (W)
WINTER = 0.6                 # share of the rated power at noon on a clear winter day
SUNRISE, SUNSET = 7.5, 17.0  # hours of the day
DUSK, DAWN = 17.5, 7.0       # the lamp is needed from dusk to dawn
P_LAMP = 40                  # power of the lamp at full light (W)
CLEARNESS = [0.9, 0.8, 0.3, 0.2, 0.25, 0.7, 0.9, 0.4, 0.3, 0.8]   # clearness of each day, from 0 to 1
S0 = 0.8                     # state of charge at midnight before the first day
DEEP = 0.15                  # below this state of charge, the battery ages fast

DT = 1 / 60                  # simulation step (h): one minute
DAYS = len(CLEARNESS)
HOURS = np.arange(0, 24 * DAYS, DT)

def p_pv(h):
    """Power of the panel (W) at the hour h since the start: a half sine wave from sunrise to sunset."""
    d, t = int(h // 24), h % 24
    if not SUNRISE <= t < SUNSET:
        return 0.0
    return P_PV_PEAK * WINTER * CLEARNESS[d] * math.sin(math.pi * (t - SUNRISE) / (SUNSET - SUNRISE))

def dark(h):
    """True from dusk to dawn."""
    t = h % 24
    return t >= DUSK or t < DAWN

def soc_step(s, p_in, p_out):
    """One step of the state of charge with the power into and out of the battery (W)."""
    net = p_in - p_out
    s += DT * (ETA_C * net if net > 0 else net / ETA_D) / E_WH
    return min(max(s, 0.0), 1.0)

def run(controller, s0=S0):
    """Ten days of the light; the controller returns the lamp power (W) from the hour and the state of charge."""
    s, log = s0, []
    for h in HOURS:
        lamp = controller(h, s) if s > 0 else 0.0     # an empty battery cannot feed the lamp
        pv = p_pv(h)
        log.append((h, s, pv, lamp, P_LAMP if dark(h) else 0.0))
        s = soc_step(s, pv, lamp)
    return pd.DataFrame(log, columns=["hour", "soc", "pv", "lamp", "need"])

def summary(df):
    """Light, darkness and the lowest state of charge of a run."""
    night = df.need > 0
    return {"light delivered (%)": float(round(100 * df.lamp[night].sum() / df.need[night].sum(), 1)),
            "dark hours": float(round(((df.lamp == 0) & night).sum() * DT, 1)),
            "lowest SoC": float(round(df.soc.min(), 3)),
            "hours of deep discharge": float(round((df.soc < DEEP).sum() * DT, 1))}

full = run(lambda h, s: P_LAMP if dark(h) else 0.0)
daily = full.assign(day=(full.hour // 24).astype(int) + 1).groupby("day").agg(
    clearness=("hour", lambda x: CLEARNESS[int(x.iloc[0] // 24)]), pv_Wh=("pv", "sum"), lamp_Wh=("lamp", "sum"),
    soc_at_midnight=("soc", "last"))
daily[["pv_Wh", "lamp_Wh"]] = (daily[["pv_Wh", "lamp_Wh"]] * DT).round(0)
print(daily.round(3).to_string())
print(summary(full))
for k, v in summary(full).items():
    RESULTS[f"full power: {k}"] = v

## 2. Modes of the controller with a low-voltage disconnect

*Lecture: [Week 3, Finite state machines](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#finite-state-machines) and [Week 3, Hybrid automata](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#hybrid-automata)*

A deep discharge damages the battery. The controller of the light is a hybrid automaton with three modes. In *Day*, the
lamp is off, and the panel charges the battery. In *Night*, the lamp is on. The mode *Protect* disconnects the lamp when
the state of charge falls to $s_{off}$. The lamp returns only after the panel has charged the battery to
$s_{on} > s_{off}$. The gap between the two thresholds is a hysteresis, so that the lamp does not flicker around a single
threshold. The class `StreetLight` keeps the mode between the minutes and records every change of mode. The cell runs
the ten days with the automaton and compares the result with Section 1.

In [ ]:
# TODO: the thresholds of the low-voltage disconnect
S_OFF = 0.20             # the lamp is disconnected at this state of charge
S_ON = 0.40              # and reconnected at this state of charge

class StreetLight:
    """The hybrid automaton of the controller with the modes Day, Night and Protect."""

    def __init__(self, dimming=None):
        self.mode, self.changes = "Day", []
        self.dimming = dimming                       # a function of the state of charge (Section 4), or None

    def __call__(self, h, s):
        old = self.mode
        if self.mode != "Protect" and s <= S_OFF:
            self.mode = "Protect"
        elif self.mode != "Protect" or s >= S_ON:
            self.mode = "Night" if dark(h) else "Day"
        if self.mode != old:
            self.changes.append((round(h, 2), old, self.mode))
        if self.mode != "Night":
            return 0.0
        return P_LAMP * (self.dimming(s) if self.dimming else 1.0)

light = StreetLight()
protected = run(light)
table = pd.DataFrame([{"controller": "full power, no protection", **summary(full)},
                      {"controller": "automaton with protection", **summary(protected)}])
print(table.to_string(index=False))
print("Changes into Protect:", [c for c in light.changes if c[2] == "Protect"])
for k, v in summary(protected).items():
    RESULTS[f"protection: {k}"] = v

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(full.hour / 24, full.soc, label="full power, no protection")
ax.plot(protected.hour / 24, protected.soc, label="automaton with protection")
ax.axhline(S_OFF, color="#B23B3B", lw=1, ls="--", label="s_off")
ax.set_xlabel("Day")
ax.set_ylabel("State of charge")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Measuring the state of charge

*Lecture: [Week 4, Sensors: From a physical quantity to a number](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#sensors) and [Week 6, The Kalman filter with matrices](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-06/lecture.html#kalman-matrix)*

The automaton needs the state of charge, which no sensor measures directly. Coulomb counting integrates the measured
power of the battery. A small offset $b$ of the current sensor adds up over the days, and the estimate drifts. The
voltage of the battery gives a second, noisy reading $z$ of the state of charge. A Kalman filter estimates the state of
charge and the offset together. With $c = \Delta t/E$ and the measured power $m_k$, its model is

$$s_{k+1} = s_k + c\,(m_k - b_k), \qquad b_{k+1} = b_k, \qquad z_k = s_k + v_k ,$$

where $v_k$ is the noise of the voltage reading. The cell runs the automaton of Section 2 with three sources of the state
of charge: Coulomb counting, the voltage reading alone and the Kalman filter. It compares their errors, the dark hours
and the hours of deep discharge.

In [ ]:
# TODO: the sensors of your battery management system
V_NOM = 12.8             # nominal voltage of the battery (V)
OFFSET_A = 0.15          # offset of the current sensor (A)
NOISE_A = 0.10           # noise of the current sensor (A)
RES_A = 20 / 1024        # resolution of the current sensor (A): 10 bits over ±10 A
SIGMA_Z = 0.05           # noise of the state of charge read from the voltage
Q_S, Q_B = 1e-8, 1e-4    # process noise of the state of charge and of the offset (W²) per step

def coulomb(s0=S0):
    state = {"s": s0}
    def step(m, z):
        state["s"] = min(max(state["s"] + m * DT / E_WH, 0.0), 1.0)
        return state["s"]
    return step

def voltage_only(s0=S0):
    return lambda m, z: z

def kalman(s0=S0):
    c = DT / E_WH
    A_m = np.array([[1.0, -c], [0.0, 1.0]])
    H = np.array([[1.0, 0.0]])
    Qm = np.diag([Q_S, Q_B])
    state = {"x": np.array([s0, 0.0]), "P": np.diag([0.05 ** 2, 10.0 ** 2])}
    def step(m, z):
        x = A_m @ state["x"] + np.array([c * m, 0.0])           # predict
        P = A_m @ state["P"] @ A_m.T + Qm
        K = P @ H.T / (H @ P @ H.T + SIGMA_Z ** 2)               # gain
        x = x + (K * (z - H @ x)).ravel()                        # correct
        state["x"], state["P"] = x, (np.eye(2) - K @ H) @ P
        return float(np.clip(x[0], 0.0, 1.0))
    step.state = state
    return step

def run_measured(controller, estimator, s0=S0, seed=3):
    """Like run, but the controller sees an estimate built from the simulated sensors."""
    rng = np.random.default_rng(seed)
    s, est, log = s0, s0, []
    for h in HOURS:
        lamp = controller(h, est) if s > 0 else 0.0
        pv = p_pv(h)
        s_new = soc_step(s, pv, lamp)
        power = (s_new - s) * E_WH / DT                                 # power into the cells (W)
        amps = power / V_NOM + OFFSET_A + rng.normal(0, NOISE_A)        # what the current sensor reads (A)
        m = RES_A * round(amps / RES_A) * V_NOM                         # measured power (W)
        z = s_new + rng.normal(0, SIGMA_Z)
        est = estimator(m, z)
        log.append((h, s, est, pv, lamp, P_LAMP if dark(h) else 0.0))
        s = s_new
    return pd.DataFrame(log, columns=["hour", "soc", "estimate", "pv", "lamp", "need"])

rows, runs = [], {}
for label, make in [("Coulomb counting", coulomb), ("voltage reading only", voltage_only), ("Kalman filter", kalman)]:
    est = make()
    df = run_measured(StreetLight(), est)
    runs[label] = df
    rmse = float(np.sqrt(np.mean((df.estimate - df.soc) ** 2)))
    rows.append({"source of the SoC": label, "RMSE of the SoC": round(rmse, 3), **summary(df)})
    if label == "Kalman filter":
        b_est = est.state["x"][1]
print(pd.DataFrame(rows).to_string(index=False))
print(f"Offset estimated by the Kalman filter after {DAYS} days: {b_est:.2f} W (true {OFFSET_A * V_NOM:.2f} W)")
RESULTS["Coulomb counting: RMSE of the SoC"] = rows[0]["RMSE of the SoC"]
RESULTS["Kalman filter: RMSE of the SoC"] = rows[2]["RMSE of the SoC"]
RESULTS["Coulomb counting: hours of deep discharge"] = rows[0]["hours of deep discharge"]
RESULTS["Kalman filter: hours of deep discharge"] = rows[2]["hours of deep discharge"]
RESULTS["estimated offset (W)"] = round(b_est, 2)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(runs["Kalman filter"].hour / 24, runs["Kalman filter"].soc, "k", lw=1, label="true SoC")
ax.plot(runs["Coulomb counting"].hour / 24, runs["Coulomb counting"].estimate, label="Coulomb counting")
ax.plot(runs["Kalman filter"].hour / 24, runs["Kalman filter"].estimate, label="Kalman filter")
ax.set_xlabel("Day")
ax.set_ylabel("State of charge")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Dimming as feedback

*Lecture: [Week 5, Feedback: Proportional control and its error](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#feedback)*

The protection of Section 2 saves the battery but leaves the street dark. Dimming spreads the energy over the nights.
A proportional controller reduces the light as the estimated state of charge $\hat{s}$ falls below a target $s_{ref}$:

$$d = \min\big(1,\ \max(d_{min},\ 1 - K_p\,(s_{ref} - \hat{s}))\big),$$

where $d$ is the share of the full power and $d_{min}$ the dimmest light that the street still needs. The lamp then
draws $d\,P_{lamp}$. With $K_p = 0$, the lamp always runs at full power. A large $K_p$ dims early and keeps the battery
high. The cell sweeps $K_p$ with the Kalman filter of Section 3 in the loop. It reports the light delivered, the dark
hours and the lowest state of charge.

In [ ]:
# TODO: the dimming of your light
S_REF = 0.9              # target state of charge: full light above it
D_MIN = 0.3              # the dimmest light that the street still needs (share of full power)
KP_DIM = 2.0             # chosen proportional gain

def dimming(kp):
    return lambda s: min(1.0, max(D_MIN, 1 - kp * (S_REF - s)))

rows = []
for kp in [0.0, 0.5, 1.0, 2.0, 4.0]:
    df = run_measured(StreetLight(dimming(kp)), kalman())
    rows.append({"Kp": kp, **summary(df)})
print(pd.DataFrame(rows).to_string(index=False))

dimmed = run_measured(StreetLight(dimming(KP_DIM)), kalman())
for k, v in summary(dimmed).items():
    RESULTS[f"dimming, Kp = {KP_DIM}: {k}"] = v

fig, ax = plt.subplots(2, 1, figsize=(8, 4.8), sharex=True)
ax[0].plot(dimmed.hour / 24, dimmed.soc, label=f"dimming, Kp = {KP_DIM}")
ax[0].plot(runs["Kalman filter"].hour / 24, runs["Kalman filter"].soc, label="no dimming")
ax[1].plot(dimmed.hour / 24, dimmed.lamp, lw=0.8)
ax[0].set_ylabel("State of charge")
ax[1].set_ylabel("Lamp power (W)")
ax[1].set_xlabel("Day")
ax[0].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. The timing of the controller tasks

*Lecture: [Week 7, Rate-monotonic scheduling](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#rate-monotonic) and [Week 7, Earliest deadline first](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#edf)*

The microcontroller of the light runs four periodic tasks, each with its deadline at the end of its period. Under
earliest deadline first (EDF), the job with the earliest absolute deadline runs, and all deadlines are met exactly when

$$U = \sum_i \frac{C_i}{T_i} \le 1 .$$

Rate-monotonic (RM) scheduling gives fixed priorities by period and guarantees less. The cell computes the response
times under RM with the iteration of Week 7. It then simulates both schedulers over one hyperperiod, the least common
multiple of the periods, and counts the missed deadlines of each task. A late job keeps running until it ends.

In [ ]:
# TODO: the periodic tasks of your controller, with execution times measured on the board (ms)
TASKS = [  # name, execution time C, period T; the deadline is the end of the period
    ("Protection: voltages and current", 2, 10),
    ("Kalman filter of the state of charge", 14, 40),
    ("Dimming control and lamp driver", 10, 50),
    ("Radio report to the city", 30, 125),
]

def rm_response(tasks):
    """Worst-case response times under RM; tasks sorted by period, None if the deadline is missed."""
    tasks = sorted(tasks, key=lambda t: t[2])
    out = []
    for i, (_, c, p) in enumerate(tasks):
        r = c
        while True:
            r_new = c + sum(math.ceil(r / pj) * cj for _, cj, pj in tasks[:i])
            if r_new == r or r_new > p:
                break
            r = r_new
        out.append(r_new)
    return out

def simulate(tasks, policy, horizon):
    """Schedule in steps of 1 ms; returns the missed deadlines of each task."""
    jobs, missed = [], [0] * len(tasks)
    for t in range(horizon):
        for i, (_, c, p) in enumerate(tasks):
            if t % p == 0:
                jobs.append({"task": i, "release": t, "deadline": t + p, "left": c, "late": False})
        for j in jobs:
            if t >= j["deadline"] and not j["late"]:
                j["late"] = True
                missed[j["task"]] += 1
        if jobs:
            if policy == "RM":
                job = min(jobs, key=lambda j: (tasks[j["task"]][2], j["release"]))
            else:
                job = min(jobs, key=lambda j: (j["deadline"], j["release"]))
            job["left"] -= 1
            if job["left"] == 0:
                jobs.remove(job)
    return missed

U = sum(c / p for _, c, p in TASKS)
horizon = math.lcm(*[p for _, _, p in TASKS])
resp = rm_response(TASKS)
rm_missed, edf_missed = simulate(TASKS, "RM", horizon), simulate(TASKS, "EDF", horizon)
table = pd.DataFrame({"task": [t[0] for t in TASKS], "C (ms)": [t[1] for t in TASKS], "T (ms)": [t[2] for t in TASKS],
                      "RM response (ms)": resp, "RM missed": rm_missed, "EDF missed": edf_missed})
print(f"Utilisation U = {U:.3f}, bound of RM for {len(TASKS)} tasks = {len(TASKS) * (2 ** (1 / len(TASKS)) - 1):.3f}")
print(f"Hyperperiod: {horizon} ms")
print(table.to_string(index=False))
RESULTS["utilisation"] = round(U, 3)
RESULTS["RM: missed deadlines in one hyperperiod"] = sum(rm_missed)
RESULTS["EDF: missed deadlines in one hyperperiod"] = sum(edf_missed)

The response time of the radio task under RM exceeds its period, so the iteration stops there. Discuss whether the radio task needs a hard deadline at all, or whether a longer period would let RM meet every deadline.

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>